# Rush 4 – Segmentation des utilisateurs d'Instagram

**Question du département Innovation** : l'usage d'Instagram est-il lié au bien-être (stress, sommeil, bonheur), et quels types d'utilisateurs les équipes produit doivent-elles distinguer ?

Ce notebook s'exécute de haut en bas sur les deux extraits fournis et produit, à chaque exécution, exactement les mêmes résultats.

**Règle de confidentialité appliquée partout** : le générateur a pu reproduire de vrais utilisateurs. Le notebook n'affiche **jamais** la ligne d'un utilisateur, et aucun tableau ne montre un groupe de moins de `EFFECTIF_MIN = 100` personnes.

| Partie | Contenu |
|---|---|
| 0 | Paramètres et chargement |
| 1 | Les colonnes : ce qui peut servir à segmenter |
| 2 | Deux extraits, lequel croire, colonne par colonne |
| 3 | Jeu de données final |
| 4 à 9 | Exploration, variables, segmentation, évaluation, limites, protection des données *(à venir)* |


## 0. Paramètres et chargement

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

GRAINE = 42            # toute opération aléatoire utilise cette graine
EFFECTIF_MIN = 100     # aucun groupe affiché sous ce seuil
DOSSIER = "data/"
F_USAGE = DOSSIER + "instagram_usage_lifestyleSafe.csv"
F_USERS = DOSSIER + "instagram_users_lifestyleSafe.csv"

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

def verifier_effectifs(table):
    """Garde-fou : refuse d'afficher un tableau d'effectifs contenant un groupe trop petit."""
    mini = int(np.nanmin(np.asarray(table, dtype=float)[np.asarray(table, dtype=float) > 0]))
    assert mini >= EFFECTIF_MIN, f"Groupe de {mini} personnes : affichage refusé"
    return table

usage = pd.read_csv(F_USAGE).sort_values("user_id").reset_index(drop=True)
users = pd.read_csv(F_USERS).sort_values("user_id").reset_index(drop=True)
print("Extrait usage :", usage.shape, "| extrait users :", users.shape)
print("Mêmes colonnes, même ordre :", list(usage.columns) == list(users.columns))
print("Mêmes identifiants :", usage["user_id"].equals(users["user_id"]))

Extrait usage : (200000, 58) | extrait users : (200000, 58)
Mêmes colonnes, même ordre : True
Mêmes identifiants : True


## 1. Les colonnes : ce qui peut servir à segmenter

L'énoncé distingue deux sources : **l'activité sur l'appli et le profil**, connus pour *tous* les utilisateurs, et **le bien-être et la santé**, connus seulement pour les membres du panel. Les segments doivent pouvoir être attribués à n'importe quel utilisateur : ils ne peuvent donc être construits qu'avec la première source.

Nous classons les 58 colonnes en cinq tiroirs. Le tiroir « profil sensible » regroupe les cas douteux : ce ne sont pas des données de santé, mais elles sont très personnelles, et nous ne les utilisons pas pour construire les segments.

In [2]:
TIROIRS = {
    "Identifiant / constante": ["user_id", "app_name"],
    "Usage de l'appli (segmentation)": [
        "daily_active_minutes_instagram", "sessions_per_day", "average_session_length_minutes",
        "posts_created_per_week", "reels_watched_per_day", "stories_viewed_per_day",
        "likes_given_per_day", "comments_written_per_day", "dms_sent_per_week", "dms_received_per_week",
        "ads_viewed_per_day", "ads_clicked_per_day", "time_on_feed_per_day", "time_on_explore_per_day",
        "time_on_messages_per_day", "time_on_reels_per_day", "followers_count", "following_count",
        "notification_response_rate", "user_engagement_score", "content_type_preference",
        "preferred_content_theme", "uses_premium_features", "subscription_status", "privacy_setting_level",
        "two_factor_auth_enabled", "biometric_login_used", "linked_accounts_count",
        "account_creation_year", "last_login_date"],
    "Profil de base (description)": ["age", "gender", "country", "urban_rural"],
    "Profil sensible (description seulement)": [
        "income_level", "employment_status", "education_level", "relationship_status",
        "has_children", "weekly_work_hours"],
    "Bien-être, santé et mode de vie (panel, évaluation seulement)": [
        "perceived_stress_score", "self_reported_happiness", "sleep_hours_per_night",
        "exercise_hours_per_week", "diet_quality", "smoking", "alcohol_frequency", "body_mass_index",
        "blood_pressure_systolic", "blood_pressure_diastolic", "daily_steps_count", "hobbies_count",
        "social_events_per_month", "books_read_per_year", "volunteer_hours_per_month",
        "travel_frequency_per_year"],
}
classees = [c for cols in TIROIRS.values() for c in cols]
assert sorted(classees) == sorted(usage.columns), "Une colonne est oubliée ou classée deux fois"
pd.DataFrame({"Tiroir": list(TIROIRS), "Nombre de colonnes": [len(v) for v in TIROIRS.values()]})

,Tiroir,Nombre de colonnes
0,Identifiant / constante,2
1,Usage de l'appli (segmentation),30
2,Profil de base (description),4
3,Profil sensible (description seulement),6
4,"Bien-être, santé et mode de vie (panel, évalua...",16


**Remarque importante** : dans la réplique, les colonnes de bien-être sont remplies pour **les 200 000 utilisateurs**, alors que l'énoncé indique qu'elles ne concernent que les membres du panel. Aucune colonne ne signale qui en fait partie. Nous traitons donc tout l'extrait comme le panel pour **mesurer** le lien usage ↔ bien-être, mais les segments eux-mêmes n'utilisent jamais ces colonnes.

## 2. Deux extraits : lequel croire, colonne par colonne

On compare les deux fichiers ligne à ligne, après les avoir triés par identifiant.

In [3]:
diff = {c: int((usage[c].astype(str) != users[c].astype(str)).sum()) for c in usage.columns}
diff = pd.Series(diff)
lignes_touchees = int((usage.astype(str) != users.astype(str)).any(axis=1).sum())
print(f"{(diff > 0).sum()} colonnes diffèrent sur {len(diff)} ; {lignes_touchees} utilisateurs concernés ({lignes_touchees/len(usage):.1%})")
diff[diff > 0].sort_values(ascending=False).to_frame("Valeurs différentes")

6 colonnes diffèrent sur 58 ; 34465 utilisateurs concernés (17.2%)


,Valeurs différentes
account_creation_year,31772
employment_status,18913
weekly_work_hours,18912
relationship_status,6623
has_children,5329
education_level,1203


Les colonnes d'usage et de bien-être sont **identiques** dans les deux extraits : le choix ne porte que sur six colonnes de profil. Pour chacune, on regarde **qui est touché** et **ce que chaque version raconte**.

In [4]:
age_touches = {c: usage.loc[usage[c].astype(str) != users[c].astype(str), "age"].agg(["min", "max", "count"])
               for c in diff[diff > 0].index}
pd.DataFrame(age_touches).T.rename(columns={"min": "âge min", "max": "âge max", "count": "utilisateurs"})

,âge min,âge max,utilisateurs
employment_status,13,17,18913
education_level,13,23,1203
relationship_status,13,17,6623
has_children,13,17,5329
weekly_work_hours,13,17,18912
account_creation_year,13,28,31772


### 2.1 Les mineurs : emploi, situation familiale, heures de travail

In [5]:
mineurs = usage["age"] < 18
print(f"Mineurs (13-17 ans) : {mineurs.sum()} utilisateurs ({mineurs.mean():.1%})")
comparaison_mineurs = pd.DataFrame({
    "Extrait usage": [
        f"{(usage.loc[mineurs, 'employment_status'] == 'Full-time employed').mean():.0%} à temps plein",
        f"{usage.loc[mineurs, 'relationship_status'].isin(['Married', 'Divorced', 'Widowed']).mean():.0%} mariés, divorcés ou veufs",
        f"{(usage.loc[mineurs, 'has_children'] == 'Yes').mean():.0%} avec enfants",
        f"{usage.loc[mineurs, 'weekly_work_hours'].mean():.0f} h de travail par semaine en moyenne"],
    "Extrait users": [
        f"{(users.loc[mineurs, 'employment_status'] == 'Not employed').mean():.0%} « Not employed »",
        f"{(users.loc[mineurs, 'relationship_status'] == 'Single').mean():.0%} célibataires",
        f"{(users.loc[mineurs, 'has_children'] == 'No').mean():.0%} sans enfant",
        f"{users.loc[mineurs, 'weekly_work_hours'].mean():.0f} h de travail par semaine"],
}, index=["Emploi", "Situation familiale", "Enfants", "Heures de travail"])
comparaison_mineurs

Mineurs (13-17 ans) : 18913 utilisateurs (9.5%)


,Extrait usage,Extrait users
Emploi,45% à temps plein,100% « Not employed »
Situation familiale,"35% mariés, divorcés ou veufs",75% célibataires
Enfants,28% avec enfants,100% sans enfant
Heures de travail,40 h de travail par semaine en moyenne,0 h de travail par semaine


**Lecture** : dans l'extrait *usage*, les mineurs ont le même profil que les adultes (45 % à temps plein, 40 h de travail par semaine, un tiers mariés, divorcés ou veufs). C'est impossible pour des 13-17 ans : le générateur a tiré ces colonnes sans tenir compte de l'âge. L'extrait *users* a corrigé ces incohérences. **Nous gardons la version users** pour ces quatre colonnes. Petite réserve : « Not employed » met dans la même case des lycéens et des chômeurs ; « Student » aurait été plus juste, mais l'information n'existe pas.

### 2.2 Le niveau d'études

In [6]:
m = usage["education_level"] != users["education_level"]
verifier_effectifs(pd.crosstab(usage.loc[m, "education_level"], users.loc[m, "education_level"]))

education_level,Bachelor,High School,Secondary
education_level,,,
PhD,299,329,575


In [7]:
print("Âge des utilisateurs concernés :", usage.loc[m, "age"].min(), "à", usage.loc[m, "age"].max(), "ans")
print("Libellés de l'extrait users :", sorted(users["education_level"].unique()))

Âge des utilisateurs concernés : 13 à 23 ans
Libellés de l'extrait users : ['Bachelor', 'Bachelor’s', 'High School', 'High school', 'Master’s', 'Other', 'PhD', 'Secondary', 'Some college']


**Lecture** : le nettoyage a retiré des doctorats déclarés par des 13-23 ans, ce qui est logique. Mais il a introduit **trois libellés nouveaux** (« Bachelor », « High School », « Secondary ») qui doublonnent les libellés existants (« Bachelor’s », « High school »). **Nous gardons la version users en harmonisant les libellés** : Bachelor → Bachelor’s, High School et Secondary → High school.

### 2.3 L'année de création du compte

In [8]:
annee_login = pd.to_datetime(usage["last_login_date"]).dt.year
diagnostic = pd.DataFrame({
    "Compte créé avant 13 ans": [int(((usage["age"] - (annee_login - x["account_creation_year"])) < 13).sum()) for x in (usage, users)],
    "Compte créé après la dernière connexion": [int((x["account_creation_year"] > annee_login).sum()) for x in (usage, users)],
}, index=["Extrait usage", "Extrait users"])
diagnostic

,Compte créé avant 13 ans,Compte créé après la dernière connexion
Extrait usage,27989,0
Extrait users,0,3724


**Lecture** : aucune des deux versions n'est fiable. L'extrait *usage* contient près de 28 000 comptes ouverts avant 13 ans, âge minimum légal pour Instagram. L'extrait *users* a « corrigé » en repoussant l'année de création, mais crée ainsi plus de 3 700 comptes ouverts **après** la dernière connexion, ce qui est impossible. **Décision : nous n'utilisons pas `account_creation_year`** (ni pour segmenter, ni pour décrire).

### 2.4 Synthèse des décisions

In [9]:
decisions = pd.DataFrame([
    ["employment_status", "users", "Mineurs à temps plein dans usage ; users les classe « Not employed »"],
    ["relationship_status", "users", "Mineurs mariés, divorcés ou veufs dans usage"],
    ["has_children", "users", "Mineurs parents dans usage"],
    ["weekly_work_hours", "users", "40 h par semaine pour des 13-17 ans dans usage"],
    ["education_level", "users, libellés harmonisés", "Doctorats avant 23 ans dans usage ; libellés en double dans users"],
    ["account_creation_year", "aucune (colonne écartée)", "Comptes créés avant 13 ans (usage) ou après la dernière connexion (users)"],
    ["52 autres colonnes", "identiques", "Aucune différence entre les deux extraits"],
], columns=["Colonne", "Version retenue", "Preuve"])
decisions

,Colonne,Version retenue,Preuve
0,employment_status,users,Mineurs à temps plein dans usage ; users les c...
1,relationship_status,users,"Mineurs mariés, divorcés ou veufs dans usage"
2,has_children,users,Mineurs parents dans usage
3,weekly_work_hours,users,40 h par semaine pour des 13-17 ans dans usage
4,education_level,"users, libellés harmonisés",Doctorats avant 23 ans dans usage ; libellés e...
5,account_creation_year,aucune (colonne écartée),Comptes créés avant 13 ans (usage) ou après la...
6,52 autres colonnes,identiques,Aucune différence entre les deux extraits


## 3. Jeu de données final

Une seule fonction, utilisée par toute l'équipe, renvoie toujours le même tableau.

In [10]:
HARMONISATION_ETUDES = {"Bachelor": "Bachelor’s", "High School": "High school", "Secondary": "High school"}

def charger_donnees():
    """Jeu final : extrait users (corrigé), libellés d'études harmonisés, colonnes inutiles retirées."""
    d = pd.read_csv(F_USERS).sort_values("user_id").reset_index(drop=True)
    d["education_level"] = d["education_level"].replace(HARMONISATION_ETUDES)
    return d.drop(columns=["app_name", "account_creation_year"])

df = charger_donnees()
print(df.shape)
print("Libellés d'études :", sorted(df["education_level"].unique()))
print("Valeurs manquantes :", int(df.isna().sum().sum()))

(200000, 56)
Libellés d'études : ['Bachelor’s', 'High school', 'Master’s', 'Other', 'PhD', 'Some college']
Valeurs manquantes : 0


### 3.1 Une colonne quasi constante

En vérifiant les distributions, une colonne de mode de vie se révèle inutilisable : le nombre de pas par jour varie à peine autour de 8 000 (98 % des valeurs entre environ 7 800 et 8 200). C'est un artefact du générateur ; nous ne l'interprétons pas.

In [11]:
df["daily_steps_count"].quantile([0.01, 0.5, 0.99]).rename(index={0.01: "1 %", 0.5: "médiane", 0.99: "99 %"}).round(0)

1 %        7794.0
médiane    8000.0
99 %       8208.0
Name: daily_steps_count, dtype: float64